<a href="https://colab.research.google.com/github/JosephBless/Deep-Learning-Colab/blob/main/Chap_7%EF%BC%9A%E5%88%A9%E7%94%A8_Backtrader_%E9%80%B2%E8%A1%8C%E7%AD%96%E7%95%A5%E5%8F%83%E6%95%B8%E5%84%AA%E5%8C%96%E5%A4%8F%E6%99%AE%E7%8E%87%E7%B4%B0%E8%AB%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

在昨天的章節中我們有先窺視了一下策略優化的事，今天，我們將仔細談談這個，我們會學習如何在 `Backtrader` 中進行多次回測，並找到最佳的策略參數配置，進一步優化我們的策略以提高夏普比率。這將使我們能夠有效地篩選出最適合的參數組合。

In [ ]:
!pip install backtrader yfinance
!pip install qgrid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.2/889.2 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached jedi-0.19.1-py2.py3-none-any.whl.metadata (22 kB)
Using cached jedi-0.19.1-py2.py3-none-any.whl (1.6 MB)
  Created wheel for qgrid: filename=qgrid-1.3.1-py2.py3-none-any.whl size=1761252 sha256=0faedfa427ad3c96428f342f37ba207389f19dab23416ce4cb4efbcfd0fa8e12
  Stored in directory: /root/.cache/pip/wheels/b2/28/9b/c1053eb92a506d814e21f415d6ce4beab694a3efed99b500bc
Successfully built qgrid


### **一、導入所需的庫**

首先，讓我們導入本範例所需的庫：

In [ ]:
import backtrader as bt
import backtrader.analyzers as btanalyzers
import pandas as pd
from datetime import datetime
import yfinance as yf  # 若要下載財經數據，可以使用 yfinance

### **二、建立策略類別並加入參數**

我們使用之前介紹過得均線交叉策略。為了執行參數最佳化，我們需要在策略類別中定義參數，如下:

In [ ]:
params = (
        ('fast_length', 10),
        ('slow_length', 50)
    )

並在指標計算中使用這些參數，像是如下:


In [ ]:
ma_fast = bt.ind.SMA(period=self.params.fast_length)
ma_slow = bt.ind.SMA(period=self.params.slow_length)

#### **1. 定義策略類別**
完整定義如下:

In [ ]:
# 建立均線交叉策略類別
class MaCrossStrategy(bt.Strategy):
    params = (
        ('fast_length', 10),
        ('slow_length', 50)
    )

    def __init__(self):
        ma_fast = bt.ind.SMA(period=self.params.fast_length)
        ma_slow = bt.ind.SMA(period=self.params.slow_length)

        self.crossover = bt.ind.CrossOver(ma_fast, ma_slow)

    def next(self):
        if not self.position:
            if self.crossover > 0:
                self.buy()
        elif self.crossover < 0:
            self.close()

### **三、設置策略參數範圍並執行優化**

#### **1. 使用 `optstrategy` 方法指定參數範圍**

In [ ]:
cerebro = bt.Cerebro()

# 使用 yfinance 下載 AAPL 的數據
data = yf.download('AAPL', start='2020-01-01', end='2021-01-01')

# 將數據轉換為 backtrader 可以使用的格式
data_bt = bt.feeds.PandasData(dataname=data)

cerebro.adddata(data_bt)

# 設定策略參數範圍
strats = cerebro.optstrategy(
    MaCrossStrategy,
    fast_length=range(1, 11),
    slow_length=range(25, 76, 5)
)

# 設定初始資金與分析工具
cerebro.broker.setcash(1000000.0)
cerebro.addsizer(bt.sizers.PercentSizer, percents=10)
cerebro.addanalyzer(btanalyzers.SharpeRatio, _name="sharpe", timeframe=bt.TimeFrame.Days, annualize=True)
cerebro.addanalyzer(btanalyzers.DrawDown, _name="drawdown")
cerebro.addanalyzer(btanalyzers.Returns, _name="returns")

# 執行回測
back = cerebro.run(maxcpus=1)

[*********************100%***********************]  1 of 1 completed


### **四、解析優化結果**

執行優化後，我們需要將結果解析成方便的格式，以便篩選出最佳的策略參數組合。基本上你將優化過程想像成網格去搜尋最佳組合。

#### **1. 提取參數與指標**

In [ ]:
# 提取回測結果中需要的參數與分析結果
par_list = []
for run in back:
    for strategy in run:
        sharpe_ratio = strategy.analyzers.sharpe.get_analysis().get('sharperatio')
        returns = strategy.analyzers.returns.get_analysis().get('rnorm100')
        drawdown = strategy.analyzers.drawdown.get_analysis().get('max', {}).get('drawdown')

        if sharpe_ratio is not None:
            par_list.append([
                strategy.params.fast_length,
                strategy.params.slow_length,
                returns,
                drawdown,
                sharpe_ratio
            ])

#### **2. 轉換為 DataFrame 並找到最佳參數**

In [ ]:
# 檢查是否有有效的結果
if not par_list:
    print("無法計算有效的夏普比率，請檢查策略或回測資料。")
else:
    # 將結果轉換成 DataFrame
    par_df = pd.DataFrame(par_list, columns=['fast_length', 'slow_length', 'return', 'drawdown', 'sharpe'])

    print("par_df:")
    print(par_df)
    print("\n")

    # 以夏普比率排序，找到最佳參數組合
    best_result = par_df.sort_values(by='sharpe', ascending=False).iloc[0]
    print(f'最佳參數組合: 快速均線={best_result["fast_length"]}, 慢速均線={best_result["slow_length"]}')
    print(f'對應的夏普比率: {best_result["sharpe"]:.2f}')

par_df:
     fast_length  slow_length    return  drawdown    sharpe
0              1           25  6.366260  2.695606  1.596513
1              1           30  6.902647  4.151988  1.499577
2              1           35  5.963844  4.601810  1.280110
3              1           40  5.738081  4.612691  1.237096
4              1           45  5.819275  4.291790  1.263973
..           ...          ...       ...       ...       ...
105           10           55  7.110246  3.545331  1.539898
106           10           60  6.681391  3.429858  1.409730
107           10           65  6.211139  3.446298  1.270534
108           10           70  5.763243  3.513028  1.164831
109           10           75  6.288409  3.375517  1.288579

[110 rows x 5 columns]


最佳參數組合: 快速均線=2.0, 慢速均線=25.0
對應的夏普比率: 1.99


### **五、結果分析與應用**
​
通過將回測結果存儲在 DataFrame 中，我們可以方便地對其進行篩選和排序，找到最適合的策略參數組合。這樣，我們就能夠有效地優化策略並提高夏普比率。
​
---
​
### **總結與作業**
​
今天我們學習了如何利用 `Backtrader` 進行策略參數的優化，並且成功找到了最佳的策略參數組合，進一步提高了策略的夏普比率。
​
**今日作業：**
1. 嘗試對不同的技術指標（如 RSI、MACD）進行策略優化。
2. 比較不同策略的夏普比率，找出最優的策略。
3. 使用 DataFrame 對結果進行可視化分析，進一步優化你的投資策略。
​
透過這些練習，你將能夠更熟練地進行策略優化，並能夠在真實市場中應用所學的量化投資技巧。

### **Appendix：動量策略 (Momentum Strategy) 的夏普比率優化實驗**

在這個附錄中，我們將進行動量策略的參數優化，並尋找出可以提高夏普比率的最佳參數組合。

---

### **Appendix-A、動量策略簡介**

動量策略是一種基於價格趨勢的交易策略，當價格持續上漲時買入，持續下跌時賣出。我們將利用 `Backtrader` 對動量策略的參數進行優化，以找到最適合的參數組合，提高夏普比率。

#### **動量策略類別定義**

In [ ]:
class MomentumStrategy(bt.Strategy):
    params = (('momentum_period', 10),)

    def __init__(self):
        # 定義動量指標
        self.momentum = bt.indicators.Momentum(self.data.close, period=self.params.momentum_period)
        self.daily_values = []  # 用於儲存每日資產價值

    def next(self):
        if not self.position:  # 如果沒有持倉
            if self.momentum[0] > 0:  # 動量指標大於0，表示上升趨勢
                self.buy()
        elif self.momentum[0] <= 0:  # 動量指標小於等於0，表示下降趨勢
            self.sell()

        # 記錄每日的資產價值
        self.daily_values.append(self.broker.getvalue())

    def stop(self):
        # 在每次策略完成時，將結果存儲到策略屬性中
        self.returns = pd.Series(self.daily_values).pct_change().dropna()
        annual_return = self.returns.mean() * 252
        annual_volatility = self.returns.std() * np.sqrt(252)
        risk_free_rate = 0.01  # 假設無風險利率為1%
        self.sharpe_ratio = (annual_return - risk_free_rate) / annual_volatility

        # 計算最大回撤
        cumulative_returns = (1 + self.returns).cumprod()
        drawdown = cumulative_returns / cumulative_returns.cummax() - 1
        self.max_drawdown = drawdown.min()

### **Appendix-B、優化動量策略參數**

接下來，我們將設定動量策略的參數範圍，並使用 `Backtrader` 進行優化，尋找出最佳的 `momentum_period` 參數以提高夏普比率。

#### **Appendix-B-1. 設定 `optstrategy` 方法**

In [ ]:
cerebro = bt.Cerebro()

# 使用 yfinance 下載 AAPL 的數據
data = yf.download('AAPL', start='2020-01-01', end='2021-01-01')

# 將數據轉換為 backtrader 可以使用的格式
data_bt = bt.feeds.PandasData(dataname=data)

cerebro.adddata(data_bt)

# 設定策略參數範圍
strats = cerebro.optstrategy(
    MomentumStrategy,
    momentum_period=range(5, 31, 5)  # 設定動量週期從5到30，每5為一個間隔
)

# 設定初始資金與分析工具
cerebro.broker.setcash(1000000.0)
cerebro.addsizer(bt.sizers.PercentSizer, percents=10)
cerebro.addanalyzer(btanalyzers.SharpeRatio, _name="sharpe", timeframe=bt.TimeFrame.Days, annualize=True)
cerebro.addanalyzer(btanalyzers.DrawDown, _name="drawdown")
cerebro.addanalyzer(btanalyzers.Returns, _name="returns")

# 執行回測
back = cerebro.run(maxcpus=1)

[*********************100%***********************]  1 of 1 completed


### **Appendix-C、解析動量策略優化結果**

我們將收集優化結果，並找出最佳的參數組合以達到最高的夏普比率。

#### **Appendix-C-1. 提取動量策略的參數與指標**

In [ ]:
# 提取回測結果中需要的參數與分析結果
par_list = []
for run in back:
    for strategy in run:
        sharpe_ratio = strategy.analyzers.sharpe.get_analysis().get('sharperatio')
        returns = strategy.analyzers.returns.get_analysis().get('rnorm100')
        drawdown = strategy.analyzers.drawdown.get_analysis().get('max', {}).get('drawdown')

        if sharpe_ratio is not None:
            par_list.append([
                strategy.params.momentum_period,
                returns,
                drawdown,
                sharpe_ratio
            ])

#### **Appendix-C-2. 轉換為 DataFrame 並找到最佳參數**


In [ ]:
# 檢查是否有有效的結果
if not par_list:
    print("無法計算有效的夏普比率，請檢查策略或回測資料。")
else:
    # 將結果轉換成 DataFrame
    par_df = pd.DataFrame(par_list, columns=['momentum_period', 'return', 'drawdown', 'sharpe'])

    # 以夏普比率排序，找到最佳參數組合
    best_result = par_df.sort_values(by='sharpe', ascending=False).iloc[0]
    print(f'最佳參數組合: 動量週期={best_result["momentum_period"]}')
    print(f'對應的夏普比率: {best_result["sharpe"]:.2f}')

最佳參數組合: 動量週期=15.0
對應的夏普比率: 1.45


### **Appendix-D、結果分析與總結**

透過這次動量策略的實驗，我們成功找到能夠最大化夏普比率的動量週期參數。這個附錄進一步展示了如何使用 `Backtrader` 進行參數優化，並將結果解析為易於分析的形式。

經由這樣的參數優化過程，我們可以更有效地提升策略表現，並在風險調整後獲得更高的超額收益。希望你能將這些方法運用在其他策略中，進一步優化你的投資決策。